 ### Zoteroize and Obsidianize a Perplexity Dialogue



 In a Perplexity dialogue copied to the clipboard by the perplexity copy button and then saved to a file, replace

 the citation numbers with matching Obsidian literature note or Zotero item links

In [1]:
import re
import pathlib as pl
import sys
from collections import defaultdict
import numpy as np
import pandas as pd
from icecream import ic
from typing import Dict, Tuple

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import link_perplexity_zotero as lpz

%load_ext autoreload
%autoreload 2

In [33]:
from matplotlib.style import available

PROMPT_END_STR_PERPLEX = '---'    
RESPONSE_SOURCES_DIVIDER_STR = '<div style="text-align: center">⁂</div>'
source_list_pattern_perplex = re.compile(r'\[\^?(?P<num>\d+)\]:\s*(?P<url>http[s]?://\S+)')

perplex_source_list_OLD_FORMATre = re.compile(r'^\[(?P<num>\d+)\]\s+(?P<url>https?://\S+)', re.M) # \n determines "end of line"
# sources_citenum_links_re = re.compile(r'\((?P<orig>\d+)\)\((?P<url>https?://[^\)]+)\)') # note used?

relinker = lpz.ZoteroLinkConverter()

# def find_first_divider_indices(markdown):
    
#     divider_start_index = markdown.find('\n---\n') # markdown divider
    
#     if divider_start_index == -1:
#         raise ValueError("No line divider '---' found in the markdown string.")

#     last_char_before_divider = divider_start_index - 1
#     first_char_after_divider = divider_start_index + len('\n---\n')
    
#     return last_char_before_divider, first_char_after_divider

class DividerNotFoundError(Exception):
    pass

def find_divider_boundaries(markdown_string):
    """ Finds the index of the last character before the markdown line divider ('---') 
    and the first character on the next line after the divider."""
    
    match = re.search(r'(?m)^---', markdown_string)
    if not match:
        raise DividerNotFoundError("'---' line divider not found")
    
    divider_start_index = match.start()
    
    # Find the index of the last character before the divider
    last_char_before_index = markdown_string.rfind('\n', 0, divider_start_index) - 1
    
    # Find where the next line starts after '---'
    first_char_after_index = markdown_string.find('\n', match.end()) + 1
    if first_char_after_index <= 0 or first_char_after_index >= len(markdown_string):
        raise DividerNotFoundError("No content found after '---' line divider")
        
    return last_char_before_index, first_char_after_index

def split_dedup_chat_text_perplex(markdown_text: str) -> Tuple[str, str, str]:
    """Splits perplexity output markdown text into prompt, response and source sections."""

    match = re.search(r'(?m)^# (?P<heading_text>.+)', markdown_text)
    if (heading_start_index := match.start('heading_text')) == -1:
        raise ValueError('Could not find prompt heading')
    
    prompt_end_index, response_start_index = find_divider_boundaries(markdown_text)
        
    if heading_start_index >= prompt_end_index:
        raise ValueError(f'{heading_start_index=} >= {prompt_end_index=}.  Probably missed the starting level 1 header part of the prompt."')
    
    prompt = markdown_text[heading_start_index:prompt_end_index+1].strip()

    response_sources_divider_index = markdown_text.rfind(RESPONSE_SOURCES_DIVIDER_STR)

    if response_sources_divider_index == -1:
        raise ValueError('Could not find divider between AI response and sources list')

    if response_sources_divider_index <= response_start_index:
        raise ValueError('body_sources_divider_index <= response_sources_divider_index')

    sources_list = f"{markdown_text[response_sources_divider_index:]}"
    source_citenum_url_pairs = []
    for m in re.finditer(source_list_pattern_perplex, sources_list):
        source_citenum_url_pairs.append((m.group('num'), m.group('url')))
    if len(source_citenum_url_pairs) < 1:
        print('Found no sources in file_text')
   
    citenums_to_url_source = relinker.citenums_to_urls_dedup(source_citenum_url_pairs)
    
    response = f"{markdown_text[response_start_index:response_sources_divider_index]}".strip()
    response_dedup = relinker.replace_body_citenums(response, citenums_to_url_source.new_num.to_dict())
    
    return prompt, response_dedup, citenums_to_url_source


def relink_perplexity_export(perplexity_file: pl.Path, relinked_file: pl.Path, verbose: bool = False) -> None:
    
    file_text = lpz.read_markdown_file(perplexity_file)
    
    prompt, response_dedup, citenums_to_url_source = split_dedup_chat_text_perplex(file_text)
    
    body_relinked, relinked_sources = relinker.relink_body_and_make_source_links(response_dedup, citenums_to_url_source)
    body_relinked = rfw.hierarch_shift_markdown_headers(body_relinked, top_level=2)

    source_link = rfw.file_link_md('source', str(perplexity_file))
    relinked_file.write_text(f'\n*{source_link}*\n# Prompt\n\n{prompt}\n# Response\n\n{body_relinked}\n# Citations\n{"\n".join(relinked_sources)}', encoding='utf-8')

## OLD FORMAT
# def dedup_and_collect_body_links_perplex_OLD_FORMAT(file_text: str, verbose: bool = False) -> tuple[str, pd.DataFrame]:
#     """Renumber links in standard Perplexity output to remove duplicates (different number, same URL). 
#     Seperate and convert the source list sting into to a deduped source list"""    

#     section_parts = file_text.split("\nCitations:\n", 1)
#     if len(section_parts) < 2:
#         print("Missing citations")
#         body, citations = section_parts, ""
#     else:
#         body, citations = section_parts

#     # Reassign body cite numbers if duplicate URLs are found in the sources
#     source_matches = list(perplex_source_list_OLD_FORMATre.finditer(citations))
#     if len(source_matches) < 1:
#         print('Found no sources in file_text')
        
#     source_citenum_url_pairs = [(match.group('num'), match.group('url')) for match in source_matches]
#     citenums_to_url_source = relinker.citenums_to_urls_dedup(source_citenum_url_pairs, verbose=verbose)
        
#     body_dedup = relinker.replace_body_citenums(body, citenums_to_url_source.new_num.to_dict())
    
#     return body_dedup, citenums_to_url_source

# def relink_perplexity_export_OLD_FORMAT(perplexity_file: pl.Path, relinked_file: pl.Path, verbose: bool = False) -> None:
#     file_text = perplexity_file.read_text(encoding='utf-8')
    
#     body_dedup, citenums_to_url = dedup_and_collect_body_links_perplex(file_text, verbose=verbose)
#     body_relinked, relinked_sources = relinker.relink_body_and_make_source_links(body_dedup, citenums_to_url)
    
#     body_relinked = rfw.hierarch_shift_markdown_headers(body_relinked, top_level=2)
#     source_link = rfw.file_link_md('source', perplexity_file)
#     relinked_file.write_text(f'\n*{source_link}*\n# Response\n{body_relinked}\n# Citations\n{"\n".join(relinked_sources)}', encoding='utf-8')

Reading from cache.


In [34]:
#perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
#perplexity_dialog_file = pl.Path(r'C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/PerPlexPro.md') # >1 for one URL
# perplexity_dialog_file = rfw.refwrangle_test_dir / 'dat' / "perple_new_format_longprompt_example.md"
perplexity_dialog_file = rfw.refwrangle_test_dir / 'dat' / 'merge_chats_perplex' / 'GPT-4o.md'


output_file = rfw.refwrangle_test_dir / 'tmp' / "tmp_perplex_example.md"
print(f'{perplexity_dialog_file=}\n-->\n{output_file=}')
verbose = False
relink_perplexity_export(perplexity_dialog_file, output_file, verbose)
print('Done.')

perplexity_dialog_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/GPT-4o.md')
-->
output_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/tmp_perplex_example.md')
Done.


### Test merging

In [35]:
tmpdir = rfw.refwrangle_test_dir / 'tmp'
tmpdir.mkdir(parents=True, exist_ok=True)

datdir = rfw.refwrangle_test_dir / 'dat' / 'merge_chats_perplex'
datdir.mkdir(parents=True, exist_ok=True)

chat_files = list(datdir.glob('*.md'))
#chat_files = [perplexity_dialog_file]

merged_output_file = tmpdir / 'tmp_stock_perplexy_merged.md'

##### Fix any duplicate cite numbers inside of each body and collect them

In [36]:
verbose = True
num_chat_files = len(chat_files)
all_prompts, all_bodies, all_citenums_to_url = [], [], []
for file_index, chat_file in enumerate(chat_files):
    if verbose:
        print(f'Parsing {chat_file.stem}')
        
    file_text = lpz.read_markdown_file(chat_file)
    
    prompt, response_dedup, citenums_to_url_source = split_dedup_chat_text_perplex(file_text)

    all_prompts.append(prompt)
    all_bodies.append(response_dedup)

    citenums_to_url_source[['file_index','chat_file']] = file_index, chat_file
    all_citenums_to_url.append(citenums_to_url_source.reset_index())
    
all_citenums_to_url = pd.concat(all_citenums_to_url)
if verbose:
    print(f'Found {len(all_citenums_to_url)} total citation numbers')

Parsing Claude 3.5 Sonnet
Parsing GPT-4o
Parsing Grok-2
Parsing Reasoning o3-mini
Parsing Reasoning R1
Parsing Sonar
Found 462 total citation numbers


In [37]:
# OLD Format

# verbose = False
# num_chat_files = len(chat_files)
# all_bodies, all_citenums_to_url = [], []
# for file_index, chat_file in enumerate(chat_files):
#     if verbose:
#         print(f'{chat_file.stem}')

#     file_text = chat_file.read_text(encoding='utf-8')
    
#     response_dedup, citenums_to_url_source = dedup_and_collect_body_links_perplex(file_text, verbose=verbose)
#     all_bodies.append(response_dedup)

#     citenums_to_url_source[['file_index','chat_file']] = file_index, chat_file
#     all_citenums_to_url.append(citenums_to_url_source.reset_index())
    
# all_citenums_to_url = pd.concat(all_citenums_to_url)
# if verbose:
#     print(f'Found {len(all_citenums_to_url)} citation numbers')

#### Make a unified cite number set for the merged document

In [38]:
# Reorder the merged citenums, giving each url a new, unique citenum.  Urls get lower 
# new citenums when they're mostly in early files and with mostly low original citenums.

# Sort the urls by the mean of the index of the files where they were used, and their citenums
df = all_citenums_to_url
df['new_num_int'] = df['new_num'].astype(int)

grouped = df.groupby('url').agg(
    mean_file_index=('file_index', 'mean'),
    mean_new_num_int=('new_num_int', 'mean')
).reset_index()

grouped = grouped.sort_values(by=['mean_file_index', 'mean_new_num_int'], 
                              ascending=True).reset_index(drop=True)

grouped['citenum_merged'] = np.arange(1, len(grouped) + 1).astype(str) # citenum == rank as sttring

# Merge back the new citenumes
df = df.merge(grouped[['url', 'citenum_merged']], on='url')

In [39]:
all_citenums_to_url = (df.sort_values('citenum_merged')
                       .rename(dict(new_num='doc_dedup_num', citenum_merged='new_num'), axis=1)
                       .drop('new_num_int', axis=1)
                       .set_index('file_index'))

all_citenums_to_url

,orig_num,doc_dedup_num,url,chat_file,new_num
file_index,,,,,
0,7,7,https://www.voanews.com/a/estonian-pm-s-party-...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,1
0,42,42,https://www.jmu.edu/news/eupolicystudies/2022/...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,10
2,74,74,https://www.euronews.com/my-europe/2024/04/23/...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,100
2,75,75,https://www.pbs.org/newshour/world/macron-le-p...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,101
2,76,76,https://influenceindustry.org/en/explorer/case...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,102
...,...,...,...,...,...
4,79,79,https://www.peoplesworld.org/article/le-pen-de...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,97
0,37,37,https://www.peoplesworld.org/article/le-pen-de...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,97
2,71,71,https://www.peoplesworld.org/article/le-pen-de...,C:\Users\scott\OneDrive\share\ref\refwrangle\t...,97


#### Assign new, unified cite numbers to each body and concatenate them into a single string

In [ ]:
# TODO: will this work for a single file?
all_prompts_same = True
for i in range(0,len(all_prompts)-1):
    is_same = all_prompts[i].strip().lower() == all_prompts[i+1].strip().lower()
    all_prompts_same &= is_same

ic(all_prompts_same);

ic| all_prompts_same: True


In [ ]:
# TODO:  

finish this bit, implementing the combinations in the obsidian table

# make a single mapping from unified citenums to urls
unified_citenums = all_citenums_to_url[['new_num', 'url']].drop_duplicates()
unified_citenums.index = unified_citenums['new_num']
unified_citenums.index.name = 'orig_num' # match expectations below TODO: needed?

response_heading = 'Responses' if len(chat_files) > 1 else 'Response'

all_bodies_unified, chat_source_file_link = '', []
for file_index, response_dedup in enumerate(all_bodies):
    if verbose:
        print(f'Unifying {chat_files[file_index].stem}')
    # remap deduped citenums to unified citenums
    citenums_to_url_this = all_citenums_to_url.loc[file_index]
    citenums_dedup_to_unified = citenums_to_url_this.set_index('doc_dedup_num').new_num.to_dict()
    body_unified = relinker.replace_body_citenums(response_dedup, citenums_dedup_to_unified) # unified citenums

    # Situate this body into the merged note context
    body_unified = rfw.hierarch_shift_markdown_headers(body_unified, top_level=2)
    source_link = rfw.file_link_md('source', chat_files[file_index])
    if num_chat_files > 1:
        all_bodies_unified += f'# {chat_files[file_index].name}\n*{source_link}*\n\n'
    else:
        all_bodies_unified = f'*{source_link}*\n# Response\n '
        
    all_bodies_unified += body_unified
    

unifying Claude 3.5 Sonnet
unifying GPT-4o
unifying Grok-2
unifying Reasoning o3-mini
unifying Reasoning R1
unifying Sonar


##### Insert links to Obsidian or Zotero

In [12]:
ic(merged_output_file)
all_bodies_unified_relinked, relinked_sources = relinker.relink_body_and_make_source_links(all_bodies_unified , unified_citenums)
relinked_sources = "\n".join(sorted(relinked_sources, key=lambda line: int(re.search(lpz.citenum_plain_re, line).group('num'))))
merged_output_file.write_text(f'{all_bodies_unified_relinked}\n# Citations\n{relinked_sources}', encoding='utf-8')
print("Done.")

ic| merged_output_file: WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/tmp/tmp_stock_perplexy_merged.md')


Done.
